In [1]:
# Данный ноутбук использовал окружение google-colab
%pip install catboost fasttext -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 3.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 8.3 MB/s eta 0:00:00


# Домашнее задание "NLP. Часть 1"

In [2]:
import math
import re
import os
import random
import json
from collections import Counter, defaultdict
from typing import List, Dict, Tuple, Any
from tqdm import tqdm

import torch
import numpy as np
import datasets
import fasttext
import fasttext.util
from transformers import BertTokenizer, BertModel

In [3]:
def seed_everything(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

In [4]:
def normalize_pretokenize_text(text: str) -> List[str]:
    text = text.lower()
    words = re.findall(r'\b\w+\b', text)
    return words

In [5]:
# This block is for tests only
test_corpus = [
    "the quick brown fox jumps over the lazy dog",
    "never jump over the lazy dog quickly",
    "brown foxes are quick and dogs are lazy"
]

def build_vocab(texts: List[str]) -> Tuple[List[str], Dict[str, int]]:
    all_words = []
    for text in texts:
        words = normalize_pretokenize_text(text)
        all_words.extend(words)
    vocab = sorted(set(all_words))
    vocab_index = {word: idx for idx, word in enumerate(vocab)}
    return vocab, vocab_index

vocab, vocab_index = build_vocab(test_corpus)

In [6]:
vocab

['and',
 'are',
 'brown',
 'dog',
 'dogs',
 'fox',
 'foxes',
 'jump',
 'jumps',
 'lazy',
 'never',
 'over',
 'quick',
 'quickly',
 'the']

## Задание 1 (0.5 балла)
Реализовать One-Hot векторизацию текстов

Вообще, One-hot векторизация производится для слов, а не для текстов. One-hot представления слов в свою очередь можно использовать в методах векторизации текстов, например, взять сумму всех векторов слов текста - Bag of words. Под One-hot векторизацией *текстов* будем понимать следующее: вектор текста имеет длину - размер словаря, а i-ый элемент вектора равен единице, если i-ый токен словаря есть в тексте (возможно более одного раза), и равен нулю, если токен отсутствует в тексте.

In [7]:
def one_hot_vectorization(text: str, vocab: List[str] = None, vocab_index: Dict[str, int] = None) -> List[int]:
    words_in_text = normalize_pretokenize_text(text)
    one_hot = []
    for token in vocab:
        if token in words_in_text:
            one_hot.append(1)
        else:
            one_hot.append(0)

    return one_hot

def test_one_hot_vectorization(corpus, vocab, vocab_index) -> bool:
    try:
        text = "the quick brown fox"
        result = one_hot_vectorization(text, vocab, vocab_index)

        if not isinstance(result, list):
            return False

        expected_length = len(vocab)
        if len(result) != expected_length:
            return False

        words_in_text = normalize_pretokenize_text(text)
        for word in words_in_text:
            if word in vocab_index:
                idx = vocab_index[word]
                if result[idx] != 1:
                    return False

        print("One-Hot-Vectors test PASSED")
        print(result)

        return True
    except Exception as e:
        print(f"One-Hot-Vectors test FAILED: {e}")
        return False

In [ ]:
assert test_one_hot_vectorization(test_corpus, vocab, vocab_index)

One-Hot-Vectors test PASSED
[0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1]


## Задание 2 (0.5 балла)
Реализовать Bag-of-Words

In [8]:
def bag_of_words_vectorization(text: str) -> Dict[str, int]:
    bag_of_words = {}
    words_in_text = normalize_pretokenize_text(text)
    for word in words_in_text:
        if word in bag_of_words:
            bag_of_words[word] += 1
        else:
            bag_of_words[word] = 1

    return bag_of_words

def test_bag_of_words_vectorization() -> bool:
    try:
        text = "the the quick brown brown brown"
        result = bag_of_words_vectorization(text)

        if not isinstance(result, dict):
            return False

        if result.get('the', 0) != 2:
            return False
        if result.get('quick', 0) != 1:
            return False
        if result.get('brown', 0) != 3:
            return False
        if result.get('nonexistent', 0) != 0:
            return False

        print("Bad-of-Words test PASSED")
        print(result)
        return True
    except Exception as e:
        print(f"Bag-of-Words test FAILED: {e}")
        return False

In [ ]:
assert test_bag_of_words_vectorization()

Bad-of-Words test PASSED
{'the': 2, 'quick': 1, 'brown': 3}


## Задание 3 (0.5 балла)
Реализовать TF-IDF

In [ ]:
def tf(token, text):
    frequency = 0
    for text_token in text:
        if (text_token == token):
            frequency += 1
    return frequency / len(text)

def idf(token, corpus):
    doc_frequency = 0
    for text in corpus:
        if token in text:
            doc_frequency += 1

    return math.log(len(corpus)/doc_frequency)

def tf_idf_vectorization(text: str, corpus: List[str] = None, vocab: List[str] = None, vocab_index: Dict[str, int] = None) -> List[float]:
    tf_idf = []
    words_in_text = normalize_pretokenize_text(text)
    words_in_text = [word for word in words_in_text if word in vocab] # Чтобы каждый токен запроса встречался хотя бы в одном из документов корпуса хотя бы один раз
    words_in_corpus_texts = []
    for corpus_text in corpus:
        words_in_corpus_texts.append(normalize_pretokenize_text(corpus_text))
    progress = 0
    for token in vocab:
        percent = progress / len(vocab)
        progress += 1
        tf_idf.append(tf(token, words_in_text)*idf(token, words_in_corpus_texts))

    return tf_idf


def test_tf_idf_vectorization(corpus, vocab, vocab_index) -> bool:
    try:
        text = "the quick brown"
        result = tf_idf_vectorization(text, corpus, vocab, vocab_index)

        if not isinstance(result, list):
            return False

        expected_length = len(vocab)
        if len(result) != expected_length:
            return False

        for val in result:
            if not isinstance(val, float):
                return False

        print("TF-IDF test PASSED")
        print(result)
        return True
    except Exception as e:
        print(f"TF-IDF test FAILED: {e}")
        return False

In [ ]:
assert test_tf_idf_vectorization(test_corpus, vocab, vocab_index)

TF-IDF test PASSED
[0.0, 0.0, 0.13515503603605478, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.13515503603605478, 0.0, 0.13515503603605478]


## Задание 4 (1 балл)
Реализовать Positive Pointwise Mutual Information (PPMI).  
https://en.wikipedia.org/wiki/Pointwise_mutual_information
$$PPMI(word, context) = max(0, PMI(word, context))$$
$$PMI(word, context) = log \frac{P(word, context)}{P(word) P(context)} = log \frac{N(word, context)|(word, context)|}{N(word) N(context)}$$
где $N(word, context)$ -- число вхождений слова $word$ в окно $context$ (размер окна -- гиперпараметр)

In [ ]:
def find_n_pairs(n, r):
    if n <= r:
        return n * (n - 1)
    elif r < n and n < 2*r:
        return (n - 1)*(2*r + 2 - n) + (n + r - 2)*(n - r - 1)
    else: # n >= 2*r
        return 2*r*(n - 2*r) + (3*r - 1)*r

def pmi(n_word, n_context, n_word_context, n_pair):
    if n_word_context == 0:
        return 0

    return math.log(n_word_context * n_pair / (n_word * n_context))

def ppmi(n_word, n_context, n_word_context, n_pair):
    return max(0, pmi(n_word, n_context, n_word_context, n_pair))

def ppmi_vectorization(
    text: str,
    corpus: List[str] = None,
    vocab: List[str] = None,
    vocab_index: Dict[str, int] = None,
    window_size: int = 2
) -> List[float]:
    words_in_text = normalize_pretokenize_text(text)
    words_in_text = [word for word in words_in_text if word in vocab] # Чтобы каждый токен запроса встречался хотя бы в одном из документов корпуса хотя бы один раз
    words_in_corpus_texts = []
    for corpus_text in corpus:
        words_in_corpus_texts.append(normalize_pretokenize_text(corpus_text))

    word_frequencies = [0]*len(vocab)

    for i in range(len(vocab)):
        for words_in_corpus_text in words_in_corpus_texts:
            for corpus_text_word in words_in_corpus_text:
                if corpus_text_word == vocab[i]:
                    word_frequencies[i] += 1

    r = window_size // 2
    # Посчитаем контекстную совстречаемость слов запроса со всеми остальными словами из словаря
    n_pairs_corpus = 0
    words_context_frequencies = []
    for text_word in words_in_text:
        word_context_frequencies = [0]*len(vocab)
        for words_in_corpus_text in words_in_corpus_texts:
            n = len(words_in_corpus_text)
            n_pairs_corpus += find_n_pairs(n, r)
            for i in range(len(words_in_corpus_text)):
                central_word = words_in_corpus_text[i]
                # Чтобы не выйти за границы текста
                lower_window_bound = max(0, i - r)
                upper_window_bound = min(n - 1, i + r)
                for j in range(lower_window_bound, upper_window_bound + 1):
                    if j != i:
                        context_word = words_in_corpus_text[j]
                        if text_word == central_word: # Если слово из запроса - центральное слово
                            word_context_frequencies[vocab_index[context_word]] += 1
                        if text_word == context_word: # Если слово из запроса - контекстное слово
                            word_context_frequencies[vocab_index[central_word]] += 1

        words_context_frequencies.append(word_context_frequencies)

    # Считаем ppmi для каждого слова из запроса для всех слов словаря
    words_ppmi = []
    for i_text_word, text_word in enumerate(words_in_text):
        n_word = word_frequencies[vocab_index[text_word]]
        word_ppmi = [0]*len(vocab)
        for i_context, context in enumerate(vocab):
            n_context = word_frequencies[vocab_index[context]]
            n_word_context = words_context_frequencies[i_text_word][i_context]
            word_ppmi[i_context] = ppmi(n_word, n_context, n_word_context, n_pairs_corpus)

        words_ppmi.append(word_ppmi)


    # Усредним полученные ppmi-векторы каждого слова, чтобы получить ppmi-вектор текста
    text_ppmi = [0]*len(vocab)
    for i in range(len(vocab)):
        sum_ppmi = 0
        for j in range(len(words_ppmi)):
            sum_ppmi += words_ppmi[j][i]
        text_ppmi[i] = sum_ppmi / len(words_ppmi)

    return text_ppmi


def test_ppmi_vectorization(corpus, vocab, vocab_index) -> bool:
    try:
        text = "quick brown fox"
        result = ppmi_vectorization(text, corpus, vocab, vocab_index)

        if not isinstance(result, list):
            return False

        expected_length = len(vocab)
        if len(result) != expected_length:
            return False

        for val in result:
            if not isinstance(val, float):
                return False

        print("PPMI test PASSED")
        print(result)
        return True
    except Exception as e:
        print(f"PPMI test FAILED: {e}")
        return False

In [ ]:
assert test_ppmi_vectorization(test_corpus, vocab, vocab_index)

PPMI test PASSED
[1.612093968983826, 1.3810449087971775, 2.9931388777810035, 0.0, 0.0, 1.612093968983826, 1.612093968983826, 0.0, 1.8431430291704745, 0.0, 0.0, 0.0, 1.3810449087971775, 0.0, 1.2458898727611227]


## Задание 5 (1 балл)
Реализовать получение эмбеддингов из fasttext и bert (для bert лучше использовать CLS токен)

In [9]:
!wget https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.bin.gz

--2025-11-04 20:45:51--  https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.bin.gz
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 3.169.121.110, 3.169.121.81, 3.169.121.107, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|3.169.121.110|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4503593528 (4.2G) [application/octet-stream]
Saving to: ‘cc.en.300.bin.gz’

cc.en.300.bin.gz    100%[===================>]   4.19G   175MB/s    in 24s     

2025-11-04 20:46:16 (177 MB/s) - ‘cc.en.300.bin.gz’ saved [4503593528/4503593528]



In [10]:
!gunzip cc.en.300.bin.gz

In [ ]:
path = 'cc.en.300.bin'
model = fasttext.load_model(path)

In [11]:
def get_fasttext_embedding(text: str, model_path: str = None, model: any = None) -> np.ndarray: # Возвращает не список массивов, а массив
    if model is None:
        if model_path is None:
            raise ValueError("Either model or model_path must be provided")
        model = fasttext.load_model(model_path)

    return model.get_sentence_vector(text)

In [ ]:
text = "This is a sample sentence."
print(get_fasttext_embedding(text, model=model))

[-3.57599487e-03 -4.72294679e-03 -3.81383449e-02  2.57753562e-02
 -5.91719151e-03 -1.65990125e-02  2.16592140e-02  5.01135830e-03
 -2.86669470e-02 -3.18915769e-03 -4.01473604e-02 -3.55030666e-03
  2.65881009e-02  3.03750392e-02 -3.95842008e-02  9.20494720e-02
  1.77136064e-03 -9.46836080e-03 -4.78756893e-03 -2.05870997e-02
 -3.32193412e-02 -2.45694574e-02  2.61960342e-03 -3.64424400e-02
  1.75956206e-03  7.91691523e-03  2.98984051e-02  1.55181205e-02
 -3.14152949e-02  8.11499357e-02 -1.91257638e-03  4.09787446e-02
  3.60504575e-02 -7.40841776e-02  5.20971743e-03  5.07564843e-03
  1.49525022e-02 -6.28905669e-02  2.94582732e-02 -2.62909569e-02
 -1.87293738e-02 -2.30435729e-02 -2.20631628e-04  9.09598358e-03
  2.51989756e-02  6.56336993e-02 -3.34903263e-02 -7.83054251e-03
 -1.48631427e-02  4.57868688e-02 -7.65839359e-03  3.17762117e-03
 -3.21611278e-02 -2.34206277e-03  2.12053843e-02  1.60637256e-02
 -1.64648574e-02 -4.43951637e-02 -6.57105148e-02  1.45856263e-02
  3.70682706e-03 -1.06735

In [21]:
_bert_model = {}
_bert_tokenizer = {}

In [12]:
def get_bert_embedding(
    text: str,
    model_name: str = 'bert-base-uncased',
    pool_method: str = 'cls'
) -> np.ndarray:
    global _bert_model, _bert_tokenizer

    if model_name not in _bert_model:
        _bert_tokenizer[model_name] = BertTokenizer.from_pretrained(model_name)
        _bert_model[model_name] = BertModel.from_pretrained(model_name)
        _bert_model[model_name].eval()

    tokenizer = _bert_tokenizer[model_name]
    model = _bert_model[model_name]

    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    )

    with torch.no_grad():
        outputs = model(**inputs)
        last_hidden = outputs.last_hidden_state  # Представления эмбеддингов после последнего энкодер блока

    token_embeddings = last_hidden[0]

    if pool_method == 'cls':
        embedding = token_embeddings[0]
    else:
        raise ValueError("pool_method must be 'cls'")

    return embedding.cpu().numpy()

In [ ]:
emb = get_bert_embedding(text)
print(emb)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

[-1.99307919e-01 -2.10060745e-01 -1.94994912e-01 -3.32378924e-01
 -5.21259606e-01 -3.73374313e-01  1.54189304e-01  4.00058061e-01
  3.53624336e-02 -1.50123447e-01 -3.52665603e-01 -2.28611216e-01
 -9.49477181e-02  1.03280000e-01  5.13516903e-01 -9.09009278e-02
  2.38635987e-01  4.82232153e-01  3.13785583e-01 -4.05656844e-01
  9.86270234e-02 -8.87676701e-02 -2.74609268e-01 -5.08711934e-01
  2.09129930e-01 -3.06265265e-01 -4.29528877e-02 -4.13104862e-01
 -4.07088138e-02  8.08255896e-02 -6.02921322e-02  3.57201695e-01
 -3.41745973e-01 -1.48240849e-01  3.90801460e-01 -1.31516352e-01
  3.67418587e-01 -9.12567787e-03  4.34040368e-01  1.41824618e-01
 -2.67848164e-01 -8.74636546e-02  2.66388267e-01  5.40870391e-02
 -5.58941178e-02 -5.74296892e-01 -2.81868815e+00 -3.43124926e-01
 -2.79367924e-01 -3.09405744e-01 -8.85440186e-02 -2.27909371e-01
  4.06301439e-01  5.09021819e-01 -2.27963597e-01  3.77305180e-01
 -2.90176153e-01  3.12225968e-01  1.94541171e-01 -1.05505601e-01
  1.51795059e-01 -6.63273

## Задание 6 (1.5 балла)
Реализовать обучение так, чтобы можно было поверх эмбеддингов, реализованных в предыдущих заданиях, обучить какую-то модель (вероятно неглубокую, например, CatBoost) на задаче классификации текстов ([IMDB](https://huggingface.co/datasets/stanfordnlp/imdb)).

К сожалению, заданный шаблон описанных выше функций подразумевает повторное вычисление статистик корпуса при каждом новом запросе, что крайне вычислительно трудоемко. Это не дает производить достаточное количество вычислений, например, для классфикации за адекватное время даже на небольшом наборе данных. BOW и One-hot кодирование не требуют корпуса текстов, поэтому вычисляются довольно быстро, а вот методы TF-IDF и PPMI придется оптимизировать.

Оптимизированный TF-IDF:

In [13]:
from collections import Counter

def compute_tf(words_in_text: List[str]) -> Dict[str, float]:
    counts = Counter(words_in_text)
    total = len(words_in_text)
    return {word: counts[word] / total for word in counts}


def compute_idf(corpus_tokenized: List[List[str]], vocab: List[str]) -> Dict[str, float]:
    doc_freq = Counter()
    for text in corpus_tokenized:
        for token in set(text):
            doc_freq[token] += 1

    total_docs = len(corpus_tokenized)
    return {token: math.log(total_docs / (1 + doc_freq[token])) for token in vocab}


def tf_idf_vectorization(
    text: str,
    idf_dict: Dict[str, float],
    vocab: List[str],
    normalize_pretokenize_text=None
) -> List[float]:
    """Создание TF-IDF вектора для одного текста."""
    words_in_text = normalize_pretokenize_text(text)
    tf_dict = compute_tf(words_in_text)
    tf_idf = [tf_dict.get(token, 0.0) * idf_dict.get(token, 0.0) for token in vocab]
    return tf_idf

Оптимизированный PPMI:

In [7]:
def find_n_pairs(n, r):
    if n <= r:
        return n * (n - 1)
    elif r < n and n < 2*r:
        return (n - 1)*(2*r + 2 - n) + (n + r - 2)*(n - r - 1)
    else: # n >= 2*r
        return 2*r*(n - 2*r) + (3*r - 1)*r

def compute_ppmi_stats(
    corpus_tokenized: List[List[str]],
    vocab: List[str],
    window_size: int = 2
) -> Dict[str, any]:
    """Предварительно вычисляем частоты и совстречаемости по всему корпусу."""
    r = window_size // 2
    vocab_index = {word: idx for idx, word in enumerate(vocab)}
    n_word = Counter()
    n_word_context = Counter()
    n_pair = 0

    for text in corpus_tokenized:
        n = len(text)
        n_pair += find_n_pairs(n, r)
        for i, center in enumerate(text):
            n_word[center] += 1
            lower = max(0, i - r)
            upper = min(n - 1, i + r)
            for j in range(lower, upper + 1):
                if j != i:
                    context = text[j]
                    n_word_context[(center, context)] += 1

    return {
        "n_word": n_word,
        "n_word_context": n_word_context,
        "n_pair": n_pair,
        "vocab_index": vocab_index
    }


def ppmi_vectorization(
    text: str,
    ppmi_stats: Dict,
    vocab: List[str],
    normalize_pretokenize_text=None
) -> List[float]:
    """Создание PPMI вектора для текста, используя заранее вычисленные статистики."""
    words_in_text = normalize_pretokenize_text(text)
    words_in_text = [w for w in words_in_text if w in vocab]
    if not words_in_text:
        return [0.0] * len(vocab)

    n_word = ppmi_stats["n_word"]
    n_word_context = ppmi_stats["n_word_context"]
    n_pair = ppmi_stats["n_pair"]
    vocab_index = ppmi_stats["vocab_index"]

    words_ppmi = []
    for word in words_in_text:
        word_ppmi = []
        n_w = n_word[word]
        for context in vocab:
            n_c = n_word[context]
            n_wc = n_word_context.get((word, context), 0)
            if n_wc == 0 or n_w == 0 or n_c == 0:
                word_ppmi.append(0.0)
            else:
                pmi = math.log((n_wc * n_pair) / (n_w * n_c))
                word_ppmi.append(max(0.0, pmi))
        words_ppmi.append(word_ppmi)

    # усредняем PPMI-векторы всех слов текста
    text_ppmi = np.mean(np.array(words_ppmi), axis=0).tolist()
    return text_ppmi

Функция векторизации текстов:

In [8]:
def vectorize_dataset(
    dataset_name: str = "imdb",
    vectorizer_type: str = "bow",
    split: str = "train",
    sample_size: int = 2500
) -> Tuple[Any, List, List]:

    dataset = datasets.load_dataset(dataset_name, split=split)

    dataset = dataset.shuffle()

    if sample_size:
        dataset = dataset.select(range(min(sample_size, len(dataset))))

    texts = [item['text'] for item in dataset if 'text' in item and item['text'].strip()]
    labels = [item['label'] for item in dataset if 'label' in item]

    def build_vocab(texts: List[str]) -> Tuple[List[str], Dict[str, int]]:
        all_words = []
        for text in texts:
            words = normalize_pretokenize_text(text)
            all_words.extend(words)
        vocab = sorted(set(all_words))
        vocab_index = {word: idx for idx, word in enumerate(vocab)}
        return vocab, vocab_index

    vocab, vocab_index = build_vocab(texts)

    tokenized_corpus = [normalize_pretokenize_text(t) for t in texts]

    idf_dict = {}
    ppmi_stats = {}
    fasttext_model = None
    if vectorizer_type == "tfidf":
        idf_dict = compute_idf(tokenized_corpus, vocab)
    elif vectorizer_type == "ppmi":
        ppmi_stats = compute_ppmi_stats(tokenized_corpus, vocab, window_size=2)
    elif vectorizer_type == "fasttext":
        fasttext_model_path = 'cc.en.300.bin'
        fasttext_model = fasttext.load_model(fasttext_model_path)

    vectorized_data = []
    for text in tqdm(texts, desc=f"Vectorizing ({vectorizer_type})", unit="text"):
        if vectorizer_type == "one_hot":
            vectorized_data.append(one_hot_vectorization(text, vocab, vocab_index))
        elif vectorizer_type == "bow":
            bow_dict = bag_of_words_vectorization(text)
            vector = [bow_dict.get(word, 0) for word in vocab]
            vectorized_data.append(vector)
        elif vectorizer_type == "tfidf":
            vectorized_data.append(tf_idf_vectorization(text, idf_dict, vocab, normalize_pretokenize_text))
        elif vectorizer_type == "ppmi":
            vectorized_data.append(ppmi_vectorization(text, ppmi_stats, vocab, normalize_pretokenize_text))
        elif vectorizer_type == "fasttext":
            embedding = get_fasttext_embedding(text, model=fasttext_model)
            vectorized_data.append(embedding.tolist())
        elif vectorizer_type == "bert":
            embedding = get_bert_embedding(text)
            vectorized_data.append(embedding.tolist())
        else:
            raise ValueError(f"Unknown vectorizer type: {vectorizer_type}")

    return vocab, vectorized_data, labels

In [9]:
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split

def train(
    embeddings_method="bow",
    test_size=0.2,
    val_size=0.2,
):
    print(f"embeddings_method - {embeddings_method}")
    vocab, X, y = vectorize_dataset("imdb", embeddings_method, "train", sample_size=2500)
    print(f"vocab size = {len(vocab)}")
    #_, X_test, y_test = vectorize_dataset("imdb", embeddings_method, "test")

    X_train, X_val_test, y_train, y_val_test = train_test_split(X, y, test_size=val_size + test_size, stratify=y)

    X_val, X_test, y_val, y_test = train_test_split(X_val_test, y_val_test, train_size= val_size / (val_size + test_size), stratify=y_val_test)

    model = CatBoostClassifier(
        iterations=1000,
        learning_rate=0.1,
        depth=3,
        loss_function='Logloss',
        eval_metric='Accuracy',
        early_stopping_rounds=50,
        verbose=100,
    )

    model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        use_best_model=True
    )

    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    print("\nTest accuracy:", acc, "\n")

    report = classification_report(y_test, y_pred, target_names=['negative', 'positive'])
    print(report)
    print()

In [17]:
train(embeddings_method="bow")

embeddings_method - bow


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Vectorizing (bow): 100%|██████████| 2500/2500 [00:04<00:00, 561.50text/s]


vocab size = 27892
0:	learn: 0.6460000	test: 0.6200000	best: 0.6200000 (0)	total: 68.4ms	remaining: 1m 8s
100:	learn: 0.8780000	test: 0.7840000	best: 0.7840000 (100)	total: 1.82s	remaining: 16.2s
200:	learn: 0.9540000	test: 0.8080000	best: 0.8080000 (200)	total: 3.54s	remaining: 14.1s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.816
bestIteration = 234

Shrink model to first 235 iterations.

Test accuracy: 0.786 

              precision    recall  f1-score   support

    negative       0.81      0.75      0.78       249
    positive       0.77      0.82      0.79       251

    accuracy                           0.79       500
   macro avg       0.79      0.79      0.79       500
weighted avg       0.79      0.79      0.79       500




In [18]:
train(embeddings_method="one_hot")

embeddings_method - one_hot


Vectorizing (one_hot): 100%|██████████| 2500/2500 [02:10<00:00, 19.20text/s]


vocab size = 28084
0:	learn: 0.6433333	test: 0.6580000	best: 0.6580000 (0)	total: 19.3ms	remaining: 19.2s
100:	learn: 0.8766667	test: 0.8260000	best: 0.8300000 (95)	total: 1.69s	remaining: 15.1s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.832
bestIteration = 106

Shrink model to first 107 iterations.

Test accuracy: 0.78 

              precision    recall  f1-score   support

    negative       0.81      0.74      0.77       254
    positive       0.75      0.83      0.79       246

    accuracy                           0.78       500
   macro avg       0.78      0.78      0.78       500
weighted avg       0.78      0.78      0.78       500




In [19]:
train(embeddings_method="tfidf")

embeddings_method - tfidf


Vectorizing (tfidf): 100%|██████████| 2500/2500 [00:12<00:00, 202.28text/s]


vocab size = 27883
0:	learn: 0.6280000	test: 0.6300000	best: 0.6300000 (0)	total: 41.3ms	remaining: 41.3s
100:	learn: 0.9166667	test: 0.8100000	best: 0.8100000 (90)	total: 3.69s	remaining: 32.8s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.828
bestIteration = 139

Shrink model to first 140 iterations.

Test accuracy: 0.788 

              precision    recall  f1-score   support

    negative       0.82      0.73      0.78       249
    positive       0.76      0.84      0.80       251

    accuracy                           0.79       500
   macro avg       0.79      0.79      0.79       500
weighted avg       0.79      0.79      0.79       500




In [11]:
train(embeddings_method="ppmi")

embeddings_method - ppmi


Vectorizing (ppmi): 100%|██████████| 2500/2500 [4:03:03<00:00,  5.83s/text]


vocab size = 28443
0:	learn: 0.6493333	test: 0.5740000	best: 0.5740000 (0)	total: 3.01s	remaining: 50m 11s
100:	learn: 0.9086667	test: 0.7260000	best: 0.7280000 (93)	total: 2m 33s	remaining: 22m 44s
200:	learn: 0.9973333	test: 0.7520000	best: 0.7560000 (195)	total: 5m 7s	remaining: 20m 21s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.756
bestIteration = 195

Shrink model to first 196 iterations.

Test accuracy: 0.77 

              precision    recall  f1-score   support

    negative       0.79      0.73      0.76       250
    positive       0.75      0.81      0.78       250

    accuracy                           0.77       500
   macro avg       0.77      0.77      0.77       500
weighted avg       0.77      0.77      0.77       500




In [24]:
train(embeddings_method="fasttext")

embeddings_method - fasttext


Vectorizing (fasttext): 100%|██████████| 2500/2500 [00:02<00:00, 1229.48text/s]


vocab size = 28332
0:	learn: 0.6933333	test: 0.7040000	best: 0.7040000 (0)	total: 21.8ms	remaining: 21.8s
100:	learn: 0.8940000	test: 0.7980000	best: 0.8120000 (85)	total: 1.13s	remaining: 10.1s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.812
bestIteration = 85

Shrink model to first 86 iterations.

Test accuracy: 0.764 

              precision    recall  f1-score   support

    negative       0.77      0.76      0.76       253
    positive       0.76      0.77      0.76       247

    accuracy                           0.76       500
   macro avg       0.76      0.76      0.76       500
weighted avg       0.76      0.76      0.76       500




In [23]:
train(embeddings_method="bert")

embeddings_method - bert


Vectorizing (bert): 100%|██████████| 2500/2500 [27:54<00:00,  1.49text/s]


vocab size = 28924
0:	learn: 0.6993333	test: 0.6580000	best: 0.6580000 (0)	total: 63.3ms	remaining: 1m 3s
100:	learn: 0.9186667	test: 0.7920000	best: 0.8020000 (85)	total: 2.92s	remaining: 26s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.806
bestIteration = 126

Shrink model to first 127 iterations.

Test accuracy: 0.852 

              precision    recall  f1-score   support

    negative       0.85      0.86      0.86       256
    positive       0.85      0.84      0.85       244

    accuracy                           0.85       500
   macro avg       0.85      0.85      0.85       500
weighted avg       0.85      0.85      0.85       500




# Выводы

* Были реализованы следующие подходы к веторизации текстов: One-hot encoding, Bag of words, TF-IDF, Positive Pointwise Mutual Information(PPMI), а также исопльзованы готовые и предобученные реализации нейросетевых подходов fasttext и BERT.
* Была произведена классификация (sentiment analysis) текстов с помощью градиентного бустинга (CatBoost) над векторными представленими текстов, полученных описанными выше методами векторизации.
* Лучшей оказалась модель BERT по всем основными метрикам (accuracy ~0.85 и f1 ~0.85), худшим подходом показал себя fasttext (accuracy ~0.76 и f1 ~0.76), остальные подходы показали себя немного лучше чем fasttext.
* Хороший результат BERT закономерен, т.к. это большая модель с удачной, как показывает практика, архитектурой, обученная на большом наборе данных. Худший результат fasttext возможно говорит о том, что его обучение происходило на текстах, контекст которых отличался от контекста отзывов на фильмы.
* Классические методы (BoW, TF-IDF, PPMI) показали промежуточные результаты (accuracy ~0.77–0.79), что свидетельствует об их достаточной эффективности для данной задачи при значительно меньших вычислительных затратах по сравнению с BERT.